## 06b — Extended Feature Extraction

Continues from `06_features.ipynb`. Adds advanced time/frequency features to the existing
80-column feature table, then splits into two datasets based on velocity regime.

### Pipeline
1. **Step 2 — Advanced Feature Extraction** (`extract_new_features`)
   - Loads existing `window_features.csv` and reconstructs raw windowed signals from `filtered_dataset.csv`
   - Overwrites `ax/ay/az kurtosis` and `ax/ay/az jerk_rms` using scipy / no-fs-scaling formulas
   - Adds 45 new columns: spectral entropy, dominant freq, spectral centroid, 40-80 Hz bandpower,
     Hjorth parameters (acc/acc_mag), kurtosis + jerk_rms + 40-80 Hz bandpower (gyro/gyro_mag),
     and three odometry statistics
   - Saves enriched table internally as `enriched_df` (125 feature cols)

2. **Step 1 — Velocity Filtering** (`create_velocity_filtered_dataset`)
   - **Dataset A**: all windows (no filtering)
   - **Dataset B**: windows where mean velocity < 0.5 m/s AND velocity std < 0.05 m/s

3. **Save** `data/processed/dataset_A_features.csv` and `dataset_B_features.csv`

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
from scipy import stats as sp_stats
from scipy import signal as sp_signal

# Make project root importable whether CWD is repo root or notebooks/
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / 'src').exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SAMPLE_RATE = 100  # Hz
ACC_COLS  = ['ax', 'ay', 'az']
GYRO_COLS = ['gx', 'gy', 'gz']
VEL_COL   = 'net_speed'

# I/O paths
features_path  = PROJECT_ROOT / 'data' / 'processed' / 'window_features.csv'
windows_path   = PROJECT_ROOT / 'data' / 'processed' / 'windowed_metadata.csv'
filtered_path  = PROJECT_ROOT / 'data' / 'interim' / 'filtered' / 'filtered_dataset.csv'
cleaned_path   = PROJECT_ROOT / 'data' / 'interim' / 'cleaned'  / 'cleaned_dataset.csv'
output_dir     = PROJECT_ROOT / 'data' / 'processed'

print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis


In [2]:
# ── Data loading ────────────────────────────────────────────────────────────
if not features_path.exists():
    raise FileNotFoundError(
        f'window_features.csv not found at {features_path}.\n'
        'Please run 06_features.ipynb first.'
    )
if not windows_path.exists():
    raise FileNotFoundError(
        f'windowed_metadata.csv not found at {windows_path}.\n'
        'Please run 05_windowing.ipynb first.'
    )

features_df = pd.read_csv(features_path, low_memory=False)
windows_df  = pd.read_csv(windows_path,  low_memory=False)

if filtered_path.exists():
    signal_path = filtered_path
    print('Using filtered dataset for feature extraction')
elif cleaned_path.exists():
    signal_path = cleaned_path
    print('Filtered dataset not found; falling back to cleaned dataset')
else:
    raise FileNotFoundError(
        'No source signal dataset found.\n'
        'Run 04_cleaning.ipynb and 04b_filtering.ipynb first.'
    )

signals_df = pd.read_csv(signal_path, low_memory=False)
signals_df = signals_df.sort_values(['run_id', 't_rel']).reset_index(drop=True)

# ── Validation ──────────────────────────────────────────────────────────────
missing_imu = [c for c in ACC_COLS + GYRO_COLS if c not in signals_df.columns]
if missing_imu:
    raise KeyError(f'Missing IMU columns in signals: {missing_imu}')
if VEL_COL not in signals_df.columns:
    raise KeyError(
        f"Velocity column '{VEL_COL}' not found. Available: {list(signals_df.columns)}"
    )
for col in ['window_id', 'run_id', 't_start', 't_end']:
    if col not in windows_df.columns:
        raise KeyError(f"Required column '{col}' missing from windowed_metadata.csv")

ID_COLS = ['window_id', 'run_id', 'segment_id', 'label', 't_start', 't_end', 'n_samples']
existing_feature_cols = [c for c in features_df.columns if c not in ID_COLS]

print(f'features_df shape:  {features_df.shape}  ({len(existing_feature_cols)} feature cols)')
print(f'windows_df  shape:  {windows_df.shape}')
print(f'signals_df  shape:  {signals_df.shape}')
print(f'Velocity column:    {VEL_COL}')

Using filtered dataset for feature extraction
features_df shape:  (1729, 87)  (80 feature cols)
windows_df  shape:  (1729, 9)
signals_df  shape:  (211705, 14)
Velocity column:    net_speed


## Step 2 — Advanced Feature Extraction

Helper functions + `extract_new_features`.

In [3]:
def band_power_welch(x, fs, f_low, f_high):
    """
    Compute band power in [f_low, f_high] Hz using Welch's PSD estimate
    and trapezoidal integration.

    Parameters
    ----------
    x : array-like  — signal samples
    fs : int        — sampling rate (Hz)
    f_low, f_high   — band edges (Hz)

    Returns
    -------
    float — band power, or np.nan for very short segments
    """
    x = np.asarray(x, dtype=float)
    if len(x) < 4:
        return np.nan
    nperseg = min(len(x), 128)
    freqs, psd = sp_signal.welch(x, fs=fs, nperseg=nperseg)
    mask = (freqs >= f_low) & (freqs <= f_high)
    if not np.any(mask):
        return 0.0
    return float(np.trapezoid(psd[mask], freqs[mask]))


def compute_spectral_features(x, fs):
    """
    Compute spectral entropy, dominant frequency, and spectral centroid
    from Welch's PSD (DC component excluded).

    Parameters
    ----------
    x : array-like  — signal samples
    fs : int        — sampling rate (Hz)

    Returns
    -------
    dict with keys: 'spectral_entropy', 'dominant_freq', 'spectral_centroid'
    """
    nan_result = {'spectral_entropy': np.nan,
                  'dominant_freq':    np.nan,
                  'spectral_centroid': np.nan}
    x = np.asarray(x, dtype=float)
    if len(x) < 4:
        return nan_result

    nperseg = min(len(x), 128)
    freqs, psd = sp_signal.welch(x, fs=fs, nperseg=nperseg)

    # Exclude DC (f == 0)
    pos_mask   = freqs > 0
    freqs_pos  = freqs[pos_mask]
    psd_pos    = psd[pos_mask]
    total_pwr  = psd_pos.sum()

    if total_pwr <= 0 or len(psd_pos) == 0:
        return nan_result

    # Spectral entropy  — treat normalised PSD as a probability distribution
    p = psd_pos / total_pwr
    with np.errstate(divide='ignore', invalid='ignore'):
        log_p = np.where(p > 0, np.log2(p), 0.0)
    spectral_entropy = float(-np.sum(p * log_p))

    # Dominant frequency  — frequency bin with maximum PSD
    dominant_freq = float(freqs_pos[np.argmax(psd_pos)])

    # Spectral centroid  — power-weighted mean frequency
    spectral_centroid = float(np.sum(freqs_pos * psd_pos) / total_pwr)

    return {
        'spectral_entropy':  spectral_entropy,
        'dominant_freq':     dominant_freq,
        'spectral_centroid': spectral_centroid,
    }


def compute_hjorth_params(x):
    """
    Compute Hjorth activity, mobility, and complexity.

    - Activity   = var(x)
    - Mobility   = std(diff(x)) / std(x)
    - Complexity = mobility(diff(x)) / mobility(x)

    Parameters
    ----------
    x : array-like — signal samples

    Returns
    -------
    dict with keys: 'hjorth_activity', 'hjorth_mobility', 'hjorth_complexity'
    """
    nan_result = {'hjorth_activity': np.nan,
                  'hjorth_mobility': np.nan,
                  'hjorth_complexity': np.nan}
    x = np.asarray(x, dtype=float)
    if len(x) < 3:
        return nan_result

    var_x  = float(np.var(x))
    std_x  = np.sqrt(var_x)

    d1     = np.diff(x)
    std_d1 = float(np.std(d1))

    activity = var_x

    if std_x < 1e-12:
        return {'hjorth_activity': activity,
                'hjorth_mobility': 0.0,
                'hjorth_complexity': 0.0}

    mobility = std_d1 / std_x

    if len(d1) < 2 or std_d1 < 1e-12:
        complexity = 0.0
    else:
        d2          = np.diff(d1)
        std_d2      = float(np.std(d2))
        mobility_d1 = std_d2 / std_d1  # mobility of 1st derivative
        complexity  = mobility_d1 / mobility if mobility > 1e-12 else 0.0

    return {
        'hjorth_activity':   activity,
        'hjorth_mobility':   float(mobility),
        'hjorth_complexity': float(complexity),
    }


print('Helper functions defined: band_power_welch, compute_spectral_features, compute_hjorth_params')

Helper functions defined: band_power_welch, compute_spectral_features, compute_hjorth_params


In [4]:
def extract_new_features(windows_df, signals_df, fs=100,
                         vel_col='net_speed', label_col='label'):
    """
    Extract additional time- and frequency-domain features from raw windowed
    IMU and odometry signals.

    Each window's raw signal is reconstructed from ``signals_df`` by matching
    ``run_id`` and the closed time interval [t_start, t_end].  Accelerometer
    channels are per-window mean-centred (gravity removal) before feature
    computation, consistent with 06_features.ipynb.

    Parameters
    ----------
    windows_df : pd.DataFrame
        Window metadata with columns: window_id, run_id, t_start, t_end.
    signals_df : pd.DataFrame
        Time-series signals with columns: run_id, t_rel, ax, ay, az,
        gx, gy, gz, and the odometry velocity column ``vel_col``.
    fs : int
        Sampling frequency in Hz (default 100).
    vel_col : str
        Column name in ``signals_df`` containing per-sample odometry velocity
        (m/s).
    label_col : str
        Name of the terrain label column (not used in computation; kept for
        compatibility).

    Returns
    -------
    feature_df : pd.DataFrame
        One row per window containing ``window_id`` and all computed feature
        columns (both overwritten and new).  Windows with no matching signal
        samples are silently skipped.

    New / overwritten columns
    -------------------------
    Accelerometer axes (ax, ay, az) — gravity-removed:
      {col}_kurtosis           [OVERWRITE]  scipy Fisher excess kurtosis
      {col}_jerk_rms           [OVERWRITE]  sqrt(mean(diff(x)^2)), no fs scaling
      {col}_spectral_entropy   [NEW]
      {col}_dominant_freq      [NEW]
      {col}_spectral_centroid  [NEW]
      {col}_bandpower_40_80Hz  [NEW]
      {col}_hjorth_activity    [NEW]
      {col}_hjorth_mobility    [NEW]
      {col}_hjorth_complexity  [NEW]

    Accelerometer magnitude (acc_mag = sqrt(ax^2+ay^2+az^2)):
      acc_mag_kurtosis, acc_mag_jerk_rms, acc_mag_spectral_entropy,
      acc_mag_dominant_freq, acc_mag_spectral_centroid,
      acc_mag_bandpower_40_80Hz,
      acc_mag_hjorth_activity, acc_mag_hjorth_mobility, acc_mag_hjorth_complexity

    Gyroscope axes (gx, gy, gz) — uncentred:
      {col}_kurtosis, {col}_jerk_rms, {col}_bandpower_40_80Hz

    Gyroscope magnitude (gyro_mag = sqrt(gx^2+gy^2+gz^2)):
      gyro_mag_kurtosis, gyro_mag_jerk_rms, gyro_mag_bandpower_40_80Hz

    Odometry:
      odom_mean_velocity  — mean(net_speed) within window (m/s)
      odom_vel_std        — std(net_speed) within window
      odom_distance       — sum(|net_speed|) / fs  (metres)
    """
    rows = []

    for _, w in windows_df.iterrows():
        run_id  = w['run_id']
        t_start = float(w['t_start'])
        t_end   = float(w['t_end'])

        seg = signals_df[
            (signals_df['run_id'] == run_id) &
            (signals_df['t_rel']  >= t_start) &
            (signals_df['t_rel']  <= t_end)
        ]
        if seg.empty:
            continue

        row = {'window_id': int(w['window_id'])}

        # ── Per-window accelerometer centering (gravity removal) ──────────
        acc_raw      = seg[ACC_COLS].to_numpy(dtype=float)
        acc_centered = acc_raw - np.mean(acc_raw, axis=0, keepdims=True)
        acc_df       = pd.DataFrame(acc_centered, columns=ACC_COLS)

        # ── Accelerometer axes ────────────────────────────────────────────
        for col in ACC_COLS:
            sig = acc_df[col].to_numpy()
            n   = len(sig)

            # Overwrite: kurtosis via scipy (Fisher / excess)
            row[f'{col}_kurtosis'] = (
                float(sp_stats.kurtosis(sig, fisher=True)) if n >= 2 else np.nan
            )

            # Overwrite: jerk RMS — no sample-rate scaling per specification
            row[f'{col}_jerk_rms'] = (
                float(np.sqrt(np.mean(np.diff(sig) ** 2))) if n >= 2 else np.nan
            )

            # New: spectral features
            sf = compute_spectral_features(sig, fs)
            row[f'{col}_spectral_entropy']  = sf['spectral_entropy']
            row[f'{col}_dominant_freq']     = sf['dominant_freq']
            row[f'{col}_spectral_centroid'] = sf['spectral_centroid']

            # New: 40-80 Hz bandpower
            row[f'{col}_bandpower_40_80Hz'] = band_power_welch(sig, fs, 40.0, 80.0)

            # New: Hjorth parameters
            hp = compute_hjorth_params(sig)
            row[f'{col}_hjorth_activity']   = hp['hjorth_activity']
            row[f'{col}_hjorth_mobility']   = hp['hjorth_mobility']
            row[f'{col}_hjorth_complexity'] = hp['hjorth_complexity']

        # ── Accelerometer magnitude ───────────────────────────────────────
        acc_mag = np.sqrt((acc_df ** 2).sum(axis=1).to_numpy())
        n_mag   = len(acc_mag)

        row['acc_mag_kurtosis'] = (
            float(sp_stats.kurtosis(acc_mag, fisher=True)) if n_mag >= 2 else np.nan
        )
        row['acc_mag_jerk_rms'] = (
            float(np.sqrt(np.mean(np.diff(acc_mag) ** 2))) if n_mag >= 2 else np.nan
        )

        sf_mag = compute_spectral_features(acc_mag, fs)
        row['acc_mag_spectral_entropy']  = sf_mag['spectral_entropy']
        row['acc_mag_dominant_freq']     = sf_mag['dominant_freq']
        row['acc_mag_spectral_centroid'] = sf_mag['spectral_centroid']
        row['acc_mag_bandpower_40_80Hz'] = band_power_welch(acc_mag, fs, 40.0, 80.0)

        hp_mag = compute_hjorth_params(acc_mag)
        row['acc_mag_hjorth_activity']   = hp_mag['hjorth_activity']
        row['acc_mag_hjorth_mobility']   = hp_mag['hjorth_mobility']
        row['acc_mag_hjorth_complexity'] = hp_mag['hjorth_complexity']

        # ── Gyroscope axes ────────────────────────────────────────────────
        for col in GYRO_COLS:
            sig = seg[col].to_numpy(dtype=float)
            n   = len(sig)

            row[f'{col}_kurtosis'] = (
                float(sp_stats.kurtosis(sig, fisher=True)) if n >= 2 else np.nan
            )
            row[f'{col}_jerk_rms'] = (
                float(np.sqrt(np.mean(np.diff(sig) ** 2))) if n >= 2 else np.nan
            )
            row[f'{col}_bandpower_40_80Hz'] = band_power_welch(sig, fs, 40.0, 80.0)

        # ── Gyroscope magnitude ───────────────────────────────────────────
        gyro_mag = np.sqrt(
            (seg[GYRO_COLS].to_numpy(dtype=float) ** 2).sum(axis=1)
        )
        n_gmag = len(gyro_mag)

        row['gyro_mag_kurtosis'] = (
            float(sp_stats.kurtosis(gyro_mag, fisher=True)) if n_gmag >= 2 else np.nan
        )
        row['gyro_mag_jerk_rms'] = (
            float(np.sqrt(np.mean(np.diff(gyro_mag) ** 2))) if n_gmag >= 2 else np.nan
        )
        row['gyro_mag_bandpower_40_80Hz'] = band_power_welch(gyro_mag, fs, 40.0, 80.0)

        # ── Odometry ──────────────────────────────────────────────────────
        vel = seg[vel_col].to_numpy(dtype=float)
        row['odom_mean_velocity'] = float(np.mean(vel))
        row['odom_vel_std']       = float(np.std(vel))
        # Distance = sum of |v_i| * dt,  dt = 1/fs (constant 100 Hz spacing)
        row['odom_distance']      = float(np.sum(np.abs(vel)) / fs)

        rows.append(row)

    return pd.DataFrame(rows)


print('extract_new_features() defined.')

extract_new_features() defined.


In [5]:
# ── Run extraction ───────────────────────────────────────────────────────────
print(f'Running extract_new_features on {len(windows_df)} windows...')
print('(Estimated time: 30-90 s at 100 Hz / 200 samples per window)')

new_features_df = extract_new_features(
    windows_df,
    signals_df,
    fs      = SAMPLE_RATE,
    vel_col = VEL_COL,
)
print(f'Done.  new_features_df shape: {new_features_df.shape}')

# ── Merge into enriched_df ───────────────────────────────────────────────────
# Columns to overwrite (same name, recomputed via scipy / updated formula)
OVERWRITE_COLS = [
    'ax_kurtosis', 'ay_kurtosis', 'az_kurtosis',
    'ax_jerk_rms', 'ay_jerk_rms', 'az_jerk_rms',
]

# Drop overwrite cols from original feature table before merging
features_base = features_df.drop(
    columns=[c for c in OVERWRITE_COLS if c in features_df.columns]
)

# Left-join on window_id: every existing window is retained
enriched_df = features_base.merge(new_features_df, on='window_id', how='left')

# Enforce column order: ID cols first, then all feature cols
id_present   = [c for c in ID_COLS if c in enriched_df.columns]
feat_present = [c for c in enriched_df.columns if c not in id_present]
enriched_df  = enriched_df[id_present + feat_present]

# ── Summary ──────────────────────────────────────────────────────────────────
n_before  = len(existing_feature_cols)
n_after   = len(feat_present)
n_overwr  = len(OVERWRITE_COLS)
n_new     = n_after - (n_before - n_overwr)

nan_counts  = enriched_df[feat_present].isna().sum()
total_nan   = int(nan_counts.sum())

print()
print('=' * 60)
print('FEATURE EXTRACTION SUMMARY')
print('=' * 60)
print(f'Features before extraction:             {n_before}')
print(f'Overwritten (recomputed, same name):    {n_overwr}')
print(f'New features added:                     {n_new}')
print(f'Features after extraction:              {n_after}')
print(f'Enriched DataFrame shape:               {enriched_df.shape}')
print(f'Total NaN values in feature cols:       {total_nan}')
if total_nan > 0:
    print('Columns with NaNs:')
    print(nan_counts[nan_counts > 0].sort_values(ascending=False).to_string())

Running extract_new_features on 1729 windows...
(Estimated time: 30-90 s at 100 Hz / 200 samples per window)
Done.  new_features_df shape: (1729, 52)

FEATURE EXTRACTION SUMMARY
Features before extraction:             80
Overwritten (recomputed, same name):    6
New features added:                     51
Features after extraction:              125
Enriched DataFrame shape:               (1729, 132)
Total NaN values in feature cols:       0


## Step 1 — Velocity Filtering

Split the enriched dataset into:
- **Dataset A** — all windows
- **Dataset B** — low constant-velocity windows (`odom_mean_velocity ≈ 0.5 m/s` AND `odom_vel_std < 0.1 m/s`)

In [6]:
def create_velocity_filtered_dataset(df, vel_col, mean_thresh=0.5,
                                     std_thresh=0.05, label_col='label'):
    """
    Split a window-level feature DataFrame into a full dataset and a
    low constant-velocity subset.

    Dataset A contains all windows.  Dataset B retains only windows
    satisfying ``vel_col < mean_thresh`` AND ``odom_vel_std < std_thresh``,
    representing near-constant low-speed locomotion that isolates terrain
    vibration from speed-dependent effects.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with one row per window.  Must contain the columns
        ``vel_col`` (per-window mean velocity) and ``odom_vel_std`` (per-window
        velocity standard deviation), both produced by ``extract_new_features``.
    vel_col : str
        Column name containing per-window mean velocity (m/s).
    mean_thresh : float
        Maximum mean velocity to keep for Dataset B (m/s).  Default 0.5.
    std_thresh : float
        Maximum velocity standard deviation for Dataset B (m/s).  Default 0.05.
    label_col : str
        Name of the terrain label column.

    Returns
    -------
    dataset_A : pd.DataFrame  — full dataset (copy)
    dataset_B : pd.DataFrame  — low constant-velocity subset (copy, index reset)
    """
    if vel_col not in df.columns:
        raise KeyError(
            f"Column '{vel_col}' not found. Run extract_new_features() first."
        )
    if 'odom_vel_std' not in df.columns:
        raise KeyError(
            "Column 'odom_vel_std' not found. Run extract_new_features() first."
        )

    dataset_A = df.copy()

    mask      = (df[vel_col] < mean_thresh) & (df['odom_vel_std'] < std_thresh)
    dataset_B = df[mask].copy().reset_index(drop=True)

    n_total   = len(df)
    n_kept    = len(dataset_B)
    n_removed = n_total - n_kept

    # ── Dataset A summary ────────────────────────────────────────────────
    print('=' * 60)
    print('DATASET A — Full Dataset')
    print('=' * 60)
    print(f'Total windows: {n_total}')
    print('Class distribution:')
    dist_a = dataset_A[label_col].value_counts().sort_index()
    for cls, cnt in dist_a.items():
        print(f'  {cls:<28} {cnt:>5}  ({cnt / n_total * 100:.1f}%)')

    # ── Dataset B summary ────────────────────────────────────────────────
    print()
    print('=' * 60)
    print('DATASET B — Low Constant-Velocity Subset')
    print(f'  Criteria: {vel_col} < {mean_thresh} m/s  AND  odom_vel_std < {std_thresh} m/s')
    print('=' * 60)
    print(f'Windows kept:    {n_kept:>5}  ({n_kept / n_total * 100:.1f}%)')
    print(f'Windows removed: {n_removed:>5}  ({n_removed / n_total * 100:.1f}%)')
    print('Class distribution:')
    dist_b = dataset_B[label_col].value_counts().sort_index()
    for cls, cnt in dist_b.items():
        pct = cnt / n_kept * 100 if n_kept > 0 else 0.0
        print(f'  {cls:<28} {cnt:>5}  ({pct:.1f}%)')

    # ── Class-level warnings ─────────────────────────────────────────────
    classes_a    = set(dist_a.index)
    classes_b    = set(dist_b.index)
    dropped      = classes_a - classes_b
    any_critical = False

    print()
    if dropped:
        print(f'  *** CRITICAL WARNING: Classes completely absent from Dataset B: '
              f'{sorted(dropped)}')
        print(f'      RECOMMENDATION: Relax thresholds '
              f'(e.g. mean_thresh={mean_thresh * 2:.2f}, '
              f'std_thresh={std_thresh * 2:.3f}).')
        any_critical = True

    for cls in sorted(classes_a):
        cnt = int(dist_b.get(cls, 0))
        if cnt < 50:
            print(f'  *** CRITICAL WARNING: \'{cls}\' has only {cnt} windows in Dataset B.')
            print(f'      This is below the minimum viable threshold of 50 windows.')
            print(f'      RECOMMENDATION: Relax velocity thresholds '
                  f'(e.g. mean_thresh={mean_thresh * 2:.2f}, '
                  f'std_thresh={std_thresh * 2:.3f}).')
            any_critical = True
        elif cnt < 100:
            print(f'  WARNING: \'{cls}\' has only {cnt} windows in Dataset B.'
                  f'  This may cause problems during model training.')

    if not any_critical and not dropped:
        all_ok = all(int(dist_b.get(cls, 0)) >= 100 for cls in classes_a)
        if all_ok:
            print('  All classes have >= 100 windows in Dataset B.')
        else:
            print('  All classes have >= 50 windows in Dataset B.')

    return dataset_A, dataset_B


print('create_velocity_filtered_dataset() defined.')

create_velocity_filtered_dataset() defined.


In [10]:
dataset_A, dataset_B = create_velocity_filtered_dataset(
    enriched_df,
    vel_col     = 'odom_mean_velocity',
    mean_thresh = 0.6,   # captures the low-speed cluster (~0.52 m/s); min in dataset is 0.476
    std_thresh  = 0.1,
    label_col   = 'label',
)

print(f'\nDataset A shape: {dataset_A.shape}')
print(f'Dataset B shape: {dataset_B.shape}')

DATASET A — Full Dataset
Total windows: 1729
Class distribution:
  dry_dirt_track                 652  (37.7%)
  grass                          386  (22.3%)
  muddy_dirt_track               145  (8.4%)
  smooth_terrain                 546  (31.6%)

DATASET B — Low Constant-Velocity Subset
  Criteria: odom_mean_velocity < 0.6 m/s  AND  odom_vel_std < 0.1 m/s
Windows kept:     1160  (67.1%)
Windows removed:   569  (32.9%)
Class distribution:
  dry_dirt_track                 523  (45.1%)
  grass                          354  (30.5%)
  muddy_dirt_track                75  (6.5%)
  smooth_terrain                 208  (17.9%)

  All classes have >= 50 windows in Dataset B.

Dataset A shape: (1729, 132)
Dataset B shape: (1160, 132)


In [11]:
# ── Save outputs ─────────────────────────────────────────────────────────────
output_dir.mkdir(parents=True, exist_ok=True)

path_a = output_dir / 'dataset_A_features.csv'
path_b = output_dir / 'dataset_B_features.csv'

dataset_A.to_csv(path_a, index=False)
dataset_B.to_csv(path_b, index=False)

# ── Final summary ─────────────────────────────────────────────────────────────
final_feat_cols = [c for c in dataset_A.columns if c not in ID_COLS]

print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)
print(f'Total feature columns in output:  {len(final_feat_cols)}')
print()
print('Dataset A — sample count per class:')
print(dataset_A['label'].value_counts().sort_index().to_string())
print()
print('Dataset B — sample count per class:')
print(dataset_B['label'].value_counts().sort_index().to_string())
print()
print('Saved files:')
print(f'  {path_a}')
print(f'  {path_b}')

FINAL SUMMARY
Total feature columns in output:  125

Dataset A — sample count per class:
label
dry_dirt_track      652
grass               386
muddy_dirt_track    145
smooth_terrain      546

Dataset B — sample count per class:
label
dry_dirt_track      523
grass               354
muddy_dirt_track     75
smooth_terrain      208

Saved files:
  /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/data/processed/dataset_A_features.csv
  /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/data/processed/dataset_B_features.csv
